# 03 (trial) — Drizzle and Inspect

**Inputs:** FLT files in `data/raw/<target>/` with WCS set by 01_download_alt or 02_tweakreg.

All drizzle parameters read from `config/wfc3_ir_drizzle_params.yaml`.

In [ ]:
# Load drizzle parameters from config — nothing hardcoded here
drizzle_cfg = yaml.safe_load(open('../config/wfc3_ir_drizzle_params.yaml'))

# AstroDrizzle lowercases the output name when constructing intermediate filenames
# (e.g. static masks). Passing a path with mixed-case directory components causes
# a FileNotFoundError. Fix: cd into the output directory first and pass only the
# base filename — all intermediate and final files are then created in the right place.
original_dir = os.getcwd()

for quasar_dir in sorted(RAW_DIR.iterdir()):
    flt_files = sorted(quasar_dir.glob('*flt.fits'))
    if not flt_files:
        continue

    target = quasar_dir.name

    # W2M1042+1641 has mixed WCS solutions (FIT_REL_GSC242 + GSC240)
    # Inspect its drizzled output carefully for smearing or doubled PSFs
    if target == 'W2M1042+1641':
        print(f'\nWARNING: {target} — mixed WCS solutions, inspect output carefully')
    else:
        print(f'\n{target}')

    # Group FLT files by filter — drizzle each filter separately
    by_filter = {}
    for flt in flt_files:
        filt = fits.getval(str(flt), 'FILTER', ext=0)
        by_filter.setdefault(filt, []).append(str(flt.resolve()))  # absolute paths for safety

    for filt, files in sorted(by_filter.items()):
        out_dir = (PROC_DIR / target / 'drizzled').resolve()
        out_dir.mkdir(parents=True, exist_ok=True)

        # Change into the output directory so AstroDrizzle writes all files there
        os.chdir(out_dir)

        output_name = f'{target}_{filt}'
        print(f'  Drizzling {filt} ({len(files)} files) → {out_dir / output_name}')

        astrodrizzle.AstroDrizzle(
            files,
            output=output_name,
            context=False,       # skip context image — not needed for science
            build=True,          # build combined multi-extension FITS output
            preserve=False,      # do not keep intermediate drizzle files
            skymethod=drizzle_cfg['skymethod'],
            driz_separate=False, # CR-rejection off — handled by up-the-ramp fitting in CALWF3
            median=False,        # CR-rejection off
            blot=False,          # CR-rejection off
            driz_cr=False,       # CR-rejection off
            final_wcs=drizzle_cfg['final_wcs'],
            final_rot=0.,        # orient output north-up
            final_scale=drizzle_cfg['final_scale'],
            final_pixfrac=drizzle_cfg['final_pixfrac'],
            final_kernel=drizzle_cfg['final_kernel'],
            driz_sep_bits=drizzle_cfg['driz_sep_bits'],
            final_bits=drizzle_cfg['final_bits'],
            combine_type=drizzle_cfg['combine_type'],
        )

        # Return to original directory before next iteration
        os.chdir(original_dir)

        # Clear verbose drizzle log output — completion message prints below
        clear_output(wait=True)
        print(f'  {target} {filt} complete')

print('\nAll quasars drizzled.')

In [ ]:
import os
import yaml
from pathlib import Path
from astropy.io import fits
from astropy import wcs
from astropy.visualization import ImageNormalize, LogStretch
import matplotlib.pyplot as plt
from drizzlepac import astrodrizzle
from IPython.display import clear_output

RAW_DIR  = Path('../data/raw')
PROC_DIR = Path('../data/processed')

In [ ]:
# Inspect drizzled science and weight images for all quasars and filters
# vmin/vmax control the display stretch — adjust per target if needed
vmin, vmax = -0.05, 100

for quasar_dir in sorted(PROC_DIR.iterdir()):
    drz_dir = quasar_dir / 'drizzled'
    drz_files = sorted(drz_dir.glob('*_drz.fits')) if drz_dir.exists() else []
    if not drz_files:
        continue

    target = quasar_dir.name

    # W2M1042+1641 has mixed WCS solutions — flag it visually before displaying
    if target == 'W2M1042+1641':
        print(f'\nWARNING: {target} — mixed WCS solutions (FIT_REL_GSC242 + GSC240)')
        print('  Check for smearing or doubled PSFs in the images below.')

    for drz_file in drz_files:
        # Parse the filter name from the filename: <TARGET>_<FILTER>_drz.fits
        filt = drz_file.stem.replace(f'{target}_', '').replace('_drz', '')

        with fits.open(drz_file) as hdu:
            # ext=1 is the drizzled science image; ext=2 is the weight map
            im_wcs = wcs.WCS(hdu[1].header)
            sci    = hdu[1].data
            wht    = hdu[2].data

        # Log stretch makes faint extended emission visible alongside the bright core
        norm = ImageNormalize(sci, vmin=vmin, vmax=vmax, stretch=LogStretch())

        fig, axes = plt.subplots(1, 2, figsize=(20, 15), subplot_kw={'projection': im_wcs})
        fig.suptitle(f'{target} — {filt}', fontsize=14)

        # Left: science image with log stretch
        axes[0].imshow(sci, norm=norm, cmap='gray', origin='lower')
        axes[0].set_title('Science (SCI)')

        # Right: weight map shows coverage — uniform weight = good dither fill
        axes[1].imshow(wht, cmap='gray', origin='lower')
        axes[1].set_title('Weight (WHT)')

        plt.tight_layout()
        plt.show()